In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
import os
import win32com.client as com

c:\Users\Roberto Ponce López\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [3]:
PATH = r"C:\Users\Roberto Ponce López\OneDrive - Instituto Tecnologico y de Estudios Superiores de Monterrey (1)\Modelación Urbana - Red Vial Guadalajara"
RED_FOLDER = r"Red Base GDL\RedBase Conectores y Atts"
SHP_FOLDER = r"red_shapefiles\11_red_final_filtrada"

In [4]:
red_base = os.path.join(PATH, RED_FOLDER, "RedBase 250826.ver")

Visum = com.Dispatch("Visum.Visum") #Visum 24 version
Visum.LoadVersion(red_base)
C = com.constants

### Merge shp-gpkg with visum

In [5]:
red_final_filter = gpd.read_file(os.path.join(PATH, SHP_FOLDER, "last", "filtered_final_network_3.gpkg"))

cols_from_shp = ["from_node", "to_node", "ageb_df_idx", "ageb_idx", "ageb_code", "topo_filt", "main_road", "connector_end_link", "network_backbone", "final_filter"]

# Read Visum Links
links_from_visum = pd.DataFrame({
    "No": [i[1] for i in Visum.Net.Links.GetMultiAttValues("No")],
    "from_node": [i[1] for i in Visum.Net.Links.GetMultiAttValues("FromNodeNo")],
    "to_node": [i[1] for i in Visum.Net.Links.GetMultiAttValues("ToNodeNo")],
    "NumTouchingLineRoutes": [i[1] for i in Visum.Net.Links.GetMultiAttValues("NumTouchingLineRoutes")],
    "GRUPO_CAP": [i[1] for i in Visum.Net.Links.GetMultiAttValues("GRUPO_CAP")],
    "HIGHWAY": [i[1] for i in Visum.Net.Links.GetMultiAttValues("HIGHWAY")]
})

# Merge cols_fro_shp to visum links
links_from_visum = links_from_visum.merge(
    red_final_filter[cols_from_shp],
    on = ["from_node", "to_node"],
    how = "left"
)

links_from_visum.head(10)

,No,from_node,to_node,NumTouchingLineRoutes,GRUPO_CAP,HIGHWAY,ageb_df_idx,ageb_idx,ageb_code,topo_filt,main_road,connector_end_link,network_backbone,final_filter
0,1.0,1.0,103492.0,0.0,MEN,motorway,1436.0,1041,1410100011041,1,7.0,1,0,1
1,1.0,103492.0,1.0,0.0,,,1436.0,1041,1410100011041,1,NaN,1,0,1
2,2.0,1.0,15708.0,1.0,ACC,motorway_link,1429.0,097A,141010001097A,1,NaN,0,0,0
3,2.0,15708.0,1.0,0.0,,,1429.0,097A,141010001097A,1,NaN,0,0,0
4,3.0,2.0,3.0,0.0,CARR,motorway,2155.0,0090,141240090,1,7.0,0,1,1
5,3.0,3.0,2.0,0.0,,,2155.0,0090,141240090,1,NaN,0,0,0
6,4.0,2.0,3068.0,0.0,CARR,motorway_link,2155.0,0090,141240090,1,NaN,0,0,0
7,4.0,3068.0,2.0,0.0,,,2155.0,0090,141240090,1,NaN,0,0,0
8,5.0,3.0,7341.0,0.0,CARR,motorway,2155.0,0090,141240090,1,7.0,0,1,1
9,5.0,7341.0,3.0,0.0,,,2155.0,0090,141240090,1,NaN,0,0,0


### Keep open links with line routes

In [6]:
mask_filter0 = links_from_visum["final_filter"] == 0
print(f"Out of {mask_filter0.sum():,} links to close")

has_lineroute = mask_filter0 & (links_from_visum["NumTouchingLineRoutes"] > 0)
print(f"{has_lineroute.sum():,} have line routes and need to be open")

# change flag of those identified to be closed BUT have a lineroute
links_from_visum.loc[has_lineroute, "final_filter"] = 1
mask_filter0_noLR = links_from_visum["final_filter"] == 0
print(f"Now {mask_filter0_noLR.sum():,} are going to be closed")

print()
print(f"{(links_from_visum["final_filter"]==1).sum():,} remain open")
print(f"Which is {((links_from_visum["final_filter"]==1).sum()/len(links_from_visum))*100:.2f}% of the network")

Out of 452,741 links to close
14,367 have line routes and need to be open
Now 438,374 are going to be closed

166,952 remain open
Which is 27.58% of the network


### Keep open links with high priority

| GRUPO_CAP | Description | 
|--------------|----------|
| CARR | Carretera / interurbana |
| ART | Arteria Urbana Principal |
| MEN | Primaria menor / lateral | 
| COL | Colectora  |
| LOC | Local          |
| ACC | Acceso / rampa       |

In [7]:
links_to_close = links_from_visum[links_from_visum["final_filter"]==0].copy()
print(f"Highway priority of {(links_from_visum["final_filter"]==0).sum():,} links to close ")
print(links_to_close["GRUPO_CAP"].value_counts())

important_highways = ["CARR", "ART", "MEN", "COL", "ACC"]
keep_open_mask = (links_from_visum["final_filter"]==0)& (links_from_visum["GRUPO_CAP"].isin(important_highways))
links_from_visum.loc[keep_open_mask, "final_filter"] = 1

print()
links_to_close = links_from_visum[links_from_visum["final_filter"]==0].copy()
print(f"Now only {(links_from_visum["final_filter"]==0).sum():,} links will be closed, and their highway priorities is:")
print(links_to_close["GRUPO_CAP"].value_counts())

print()
print(f"Which means {(links_from_visum["final_filter"]==1).sum():,} will remain open")
print(f"Keeping {((links_from_visum["final_filter"]==1).sum()/len(links_from_visum))*100:.2f}% of the network")

Highway priority of 438,374 links to close 
GRUPO_CAP
        241363
LOC     180074
COL       8942
MEN       2706
ART       2467
CARR      2149
ACC        673
Name: count, dtype: int64

Now only 421,437 links will be closed, and their highway priorities is:
GRUPO_CAP
       241363
LOC    180074
Name: count, dtype: int64

Which means 183,889 will remain open
Keeping 30.38% of the network


In [8]:
pd.set_option("display.max_rows", None)
print(f"{(links_from_visum["final_filter"]==0).sum():,} links to close ")
print(f"But keep open those of OSM main links (primary, secondary, tertiary, trunk, motorway)")


highways_to_keep = ["primary", "primary_link", "secondary", "secondary_link", "tertiary", "tertiary_link", "motorway", "motorway_link", "trunk", "trunk_link"]
keep_open_mask = (links_from_visum['final_filter']==0) & (links_from_visum['HIGHWAY'].isin(highways_to_keep))
links_from_visum.loc[keep_open_mask, "final_filter"] = 1

print()
print(f"Now only {(links_from_visum["final_filter"]==0).sum():,} links will be closed, and their OSM highways are:")
links_to_close = links_from_visum[links_from_visum["final_filter"]==0].copy()
print(links_to_close["HIGHWAY"].value_counts())

print()
print(f"Which means {(links_from_visum["final_filter"]==1).sum():,} will remain open")
print(f"Keeping {((links_from_visum["final_filter"]==1).sum()/len(links_from_visum))*100:.2f}% of the network")

421,437 links to close 
But keep open those of OSM main links (primary, secondary, tertiary, trunk, motorway)

Now only 417,995 links will be closed, and their OSM highways are:
HIGHWAY
residential                                       193820
service                                            68453
                                                   62579
living_street                                      45930
footway                                            19412
unclassified                                       10399
path                                                5783
cycleway                                            3456
track                                               2809
pedestrian                                          2487
steps                                                486
['path', 'residential']                              327
['footway', 'steps']                                 279
['footway', 'residential']                           276
['living_street'

### Add final filter in Visum
- final_filter == 0 (links to close)
- links to close don't include links with line routes
- links to close don't include links with high priorities (daniel's tags)

In [10]:
visum_links = Visum.Net.Links

links =  links_from_visum.copy()
links = links.reset_index(drop=True)
links.index = links.index + 1

attributes_gpkg = {
    "FINAL_FILTER": ("final_filter", int),  
}
 

for visum_att, (df_col, dtype) in attributes_gpkg.items():
    values = list(
        zip(
            links.index,
            links[df_col].astype(dtype)
        )
    )

    visum_links.SetMultiAttValues(visum_att, values)